# ImageNet + ResNet-50: Smoothing Radius vs Case Align and Sensitivity

This notebook runs the same smoothing-radius experiment on:

- dataset: local ImageNet-style validation split loaded via `ImageFolder`
- model: pretrained `torchvision` `ResNet-50`
- explainer: `Integrated Gradients`
- problem metric: `cosine`
- solution metric: `cosine`
- Case Align neighbourhood size: `k=25`

Expected dataset layout:

```text
src/data/imagenet/val/<class_folder>/<image files>
```

where `<class_folder>` is the standard ImageNet synset folder name such as `n01440764`.

In [ ]:
import json
import ssl
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import torch
import torch.nn.functional as F
from captum.attr import IntegratedGradients
from captum.metrics import sensitivity_max
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, models

current_dir = Path.cwd()
if current_dir.name == "experiments":
    SRC_DIR = current_dir.parent
elif (current_dir / "src").exists():
    SRC_DIR = current_dir / "src"
else:
    SRC_DIR = current_dir

if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

from case_align.metrics import safe_normalise_rows
from train_mnist_model import set_seed

DATA_DIR = SRC_DIR / "data"
IMAGENET_VAL_DIR = DATA_DIR / "imagenet" / "val"
OUTPUT_DIR = SRC_DIR / "results" / "imagenet_resnet50_smoothing_radius_sensitivity_vs_case_align_cosine_k25"

METHODS = ["ig"]
K = 25
PROBLEM_METRIC = "cosine"
SOLUTION_METRIC = "cosine"
SMOOTHING_RADII = np.round(np.linspace(0.1, 2.0, 15), 2)

PERTURB_RADIUS = 0.1
N_PERTURB_SAMPLES = 5
RETRIEVAL_BATCH_SIZE = 32
MAX_RETRIEVAL_SAMPLES = 1000
MAX_QUERY_SAMPLES = 30
SEED = 42

METHOD_LABELS = {"ig": "Integrated Gradients"}

print(f"ImageNet val dir: {IMAGENET_VAL_DIR}")
print(f"Radii: {SMOOTHING_RADII[0]} to {SMOOTHING_RADII[-1]} ({len(SMOOTHING_RADII)} values)")
print(f"k={K}, retrieval cap={MAX_RETRIEVAL_SAMPLES}, query cap={MAX_QUERY_SAMPLES}")

In [ ]:
def configure_ssl_for_macos() -> None:
    ssl._create_default_https_context = ssl._create_unverified_context


def load_resnet50(device: torch.device):
    weights = models.ResNet50_Weights.DEFAULT
    model = models.resnet50(weights=weights)
    model.eval()
    model.to(device)
    return model, weights


def load_imagenet_subset(imagenet_val_dir: Path, weights, max_retrieval_samples=None, seed=42):
    if not imagenet_val_dir.exists():
        raise FileNotFoundError(
            f"ImageNet validation directory not found: {imagenet_val_dir}. "
            "Expected ImageFolder layout under src/data/imagenet/val/."
        )

    transform = weights.transforms()
    dataset = datasets.ImageFolder(str(imagenet_val_dir), transform=transform)

    if max_retrieval_samples is not None and len(dataset) > max_retrieval_samples:
        rng = np.random.default_rng(seed)
        indices = np.sort(rng.choice(len(dataset), size=max_retrieval_samples, replace=False))
        dataset = Subset(dataset, indices.tolist())

    loader = DataLoader(dataset, batch_size=64, shuffle=False, num_workers=0)
    image_chunks, label_chunks = [], []
    for images, labels in loader:
        image_chunks.append(images)
        label_chunks.append(labels)

    images_tensor = torch.cat(image_chunks, dim=0).float()
    labels_np = torch.cat(label_chunks, dim=0).numpy().astype(int)
    return images_tensor, labels_np, dataset


def predict_labels_for_images(model, images, batch_size, device):
    pred_chunks, conf_chunks = [], []
    with torch.no_grad():
        for start in range(0, len(images), batch_size):
            end = min(start + batch_size, len(images))
            logits = model(images[start:end].to(device))
            probs = torch.softmax(logits, dim=1)
            conf, preds = probs.max(dim=1)
            pred_chunks.append(preds.detach().cpu().numpy().astype(int))
            conf_chunks.append(conf.detach().cpu().numpy().astype(float))
    return np.concatenate(pred_chunks), np.concatenate(conf_chunks)


def select_query_indices(retrieval_labels, max_query_samples, seed=42):
    rng = np.random.default_rng(seed)
    query_indices = []
    for label in sorted(np.unique(retrieval_labels)):
        label_indices = np.where(retrieval_labels == label)[0]
        if label_indices.size > 1:
            query_indices.append(int(rng.choice(label_indices)))
    remaining = max(0, max_query_samples - len(query_indices))
    pool = np.setdiff1d(np.arange(len(retrieval_labels)), np.array(query_indices, dtype=int))
    if remaining > 0 and pool.size > 0:
        extra = rng.choice(pool, size=min(remaining, pool.size), replace=False)
        query_indices.extend(extra.tolist())
    return np.array(sorted(query_indices[:max_query_samples]), dtype=int)


def cosine_distance_context(matrix, epsilon=1e-8):
    matrix_raw = np.asarray(matrix, dtype=float)
    return {
        "matrix_raw": matrix_raw,
        "matrix_repr": safe_normalise_rows(matrix_raw, eps=epsilon),
        "epsilon": float(epsilon),
    }


def cosine_row_distances(query_vec, context):
    query_repr = safe_normalise_rows(np.asarray(query_vec, dtype=float)[None, :], eps=context["epsilon"])[0]
    similarity = context["matrix_repr"] @ query_repr
    return np.clip(1.0 - 0.5 * (similarity + 1.0), 0.0, 1.0)


def compute_case_align_like_only(query_index, query_label, retrieval_labels, problem_context, solution_context, k, epsilon=1e-8):
    query_problem = problem_context["matrix_raw"][query_index]
    dprob_all = cosine_row_distances(query_problem, problem_context)
    like_mask = retrieval_labels == int(query_label)
    like_mask[query_index] = False
    like_indices = np.where(like_mask)[0]
    if like_indices.size == 0:
        return np.nan, 0, False
    like_dists = dprob_all[like_indices]
    k_use = min(int(k), like_indices.size)
    neigh_indices = like_indices[np.argsort(like_dists)[:k_use]]
    dprob_neigh = dprob_all[neigh_indices]
    query_solution = solution_context["matrix_raw"][query_index]
    dsoln_all = cosine_row_distances(query_solution, solution_context)
    dsoln_neigh = dsoln_all[neigh_indices]
    denom = max(float(np.max(dsoln_all) - np.min(dsoln_all)), epsilon)
    align = 1.0 - (dsoln_neigh - float(np.min(dsoln_all))) / denom
    weights = 1.0 - dprob_neigh
    weight_sum = float(np.sum(weights))
    if weight_sum <= 0:
        return 0.0, int(k_use), True
    return float(np.sum(weights * align) / weight_sum), int(k_use), True


def gaussian_kernel2d(radius, device, dtype):
    sigma = float(radius)
    kernel_size = max(3, int(2 * np.ceil(3 * sigma) + 1))
    coords = torch.arange(kernel_size, device=device, dtype=dtype) - (kernel_size - 1) / 2
    g = torch.exp(-(coords ** 2) / (2 * sigma ** 2))
    g = g / g.sum()
    kernel = torch.outer(g, g)
    kernel = kernel / kernel.sum()
    return kernel.unsqueeze(0).unsqueeze(0)


def smooth_attributions(attrs, radius):
    kernel = gaussian_kernel2d(radius, attrs.device, attrs.dtype)
    kernel_size = kernel.shape[-1]
    channels = attrs.shape[1]
    kernel = kernel.expand(channels, 1, kernel_size, kernel_size)
    return F.conv2d(attrs, kernel, padding=kernel_size // 2, groups=channels)


def attribute_batch(ig, images, targets, baseline, device):
    xb = images.to(device).clone().detach().requires_grad_(True)
    target_tensor = torch.as_tensor(targets, dtype=torch.long, device=device)
    return ig.attribute(xb, baselines=baseline.to(device).expand_as(xb), target=target_tensor)


def compute_retrieval_attributions(ig, images, pred_labels, baseline, batch_size, device):
    chunks = []
    n_batches = (len(images) + batch_size - 1) // batch_size
    for batch_idx, start in enumerate(range(0, len(images), batch_size), start=1):
        end = min(start + batch_size, len(images))
        attrs = attribute_batch(ig, images[start:end], pred_labels[start:end], baseline, device)
        chunks.append(attrs.detach().cpu().float())
        if batch_idx == 1 or batch_idx == n_batches or batch_idx % max(n_batches // 10, 1) == 0:
            print(f"      attribution batches: {batch_idx}/{n_batches}")
    return torch.cat(chunks, dim=0)


def compute_sensitivity_for_radius(ig, image, target, baseline, perturb_radius, n_perturb_samples, device, smoothing_radius):
    x = image.unsqueeze(0).to(device)
    b0 = baseline.to(device)
    def explain_func(inputs, target=None):
        x_in = inputs[0] if isinstance(inputs, tuple) else inputs
        attrs = ig.attribute(x_in, baselines=b0.expand_as(x_in), target=target)
        return smooth_attributions(attrs, radius=smoothing_radius)
    sens = sensitivity_max(
        explanation_func=explain_func,
        inputs=x,
        target=int(target),
        perturb_radius=perturb_radius,
        n_perturb_samples=n_perturb_samples,
    )
    return float(sens.detach().cpu().item())

In [ ]:
set_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model, weights = load_resnet50(device)
ig = IntegratedGradients(model)

retrieval_images, retrieval_labels, dataset_obj = load_imagenet_subset(
    IMAGENET_VAL_DIR,
    weights=weights,
    max_retrieval_samples=MAX_RETRIEVAL_SAMPLES,
    seed=SEED,
)
retrieval_pred_labels, retrieval_confidences = predict_labels_for_images(
    model=model,
    images=retrieval_images,
    batch_size=RETRIEVAL_BATCH_SIZE,
    device=device,
)

query_indices = select_query_indices(retrieval_labels, MAX_QUERY_SAMPLES, seed=SEED)
query_images = retrieval_images[query_indices].clone()
query_labels = retrieval_labels[query_indices].copy()
query_pred_labels = retrieval_pred_labels[query_indices].copy()
query_confidences = retrieval_confidences[query_indices].copy()

baseline = torch.zeros((1, *retrieval_images.shape[1:]), dtype=retrieval_images.dtype)
X_flat = retrieval_images.view(len(retrieval_images), -1).detach().cpu().numpy()
problem_context = cosine_distance_context(X_flat)

print(f"Retrieval pool size: {len(retrieval_images)}")
print(f"Query samples: {len(query_images)}")
print(f"Unique labels among queries: {len(np.unique(query_labels))}")

## Run Smoothing-Radius Sweep

In [ ]:
score_rows = []

print("Computing unsmoothed retrieval attributions once...")
raw_retrieval_attrs = compute_retrieval_attributions(
    ig=ig,
    images=retrieval_images,
    pred_labels=retrieval_pred_labels,
    baseline=baseline,
    batch_size=RETRIEVAL_BATCH_SIZE,
    device=device,
)

for radius in SMOOTHING_RADII:
    radius = float(radius)
    print(f"Radius {radius:.2f}: Case Align + sensitivity")
    smoothed_attrs = smooth_attributions(raw_retrieval_attrs.to(device), radius=radius).detach().cpu()
    E_flat = smoothed_attrs.view(len(retrieval_images), -1).numpy()
    solution_context = cosine_distance_context(E_flat)

    for i in range(len(query_images)):
        retrieval_index = int(query_indices[i])
        s_plus, like_count, has_like_neighbour = compute_case_align_like_only(
            query_index=retrieval_index,
            query_label=int(query_labels[i]),
            retrieval_labels=retrieval_labels,
            problem_context=problem_context,
            solution_context=solution_context,
            k=K,
        )
        sensitivity = compute_sensitivity_for_radius(
            ig=ig,
            image=query_images[i],
            target=int(query_pred_labels[i]),
            baseline=baseline,
            perturb_radius=PERTURB_RADIUS,
            n_perturb_samples=N_PERTURB_SAMPLES,
            device=device,
            smoothing_radius=radius,
        )
        score_rows.append({
            "method": "ig",
            "method_label": METHOD_LABELS["ig"],
            "smoothing_radius": radius,
            "sample_position": int(i),
            "original_test_index": retrieval_index,
            "true_label": int(query_labels[i]),
            "pred_label": int(query_pred_labels[i]),
            "confidence": float(query_confidences[i]),
            "case_align_like_count": int(like_count),
            "case_align_has_like_neighbour": bool(has_like_neighbour),
            "case_align_S_plus": float(s_plus) if has_like_neighbour else np.nan,
            "captum_sensitivity": float(sensitivity),
        })

scores_df = pd.DataFrame(score_rows)
summary_df = (
    scores_df.groupby(["method", "method_label", "smoothing_radius"], as_index=False)
    .agg(
        n_samples=("sample_position", "count"),
        mean_case_align_S_plus=("case_align_S_plus", "mean"),
        std_case_align_S_plus=("case_align_S_plus", "std"),
        mean_captum_sensitivity=("captum_sensitivity", "mean"),
        std_captum_sensitivity=("captum_sensitivity", "std"),
    )
)

display(summary_df.round(5))

In [ ]:
plot_df = summary_df.melt(
    id_vars=["method", "method_label", "smoothing_radius"],
    value_vars=["mean_case_align_S_plus", "mean_captum_sensitivity"],
    var_name="metric",
    value_name="mean_value",
)
plot_df["metric"] = plot_df["metric"].map({
    "mean_case_align_S_plus": "Case Align S_plus",
    "mean_captum_sensitivity": "Captum sensitivity_max",
})
plot_df["series"] = plot_df["method_label"] + " - " + plot_df["metric"]
fig = px.line(
    plot_df,
    x="smoothing_radius",
    y="mean_value",
    color="series",
    markers=True,
    title="ImageNet ResNet-50: smoothing radius vs sensitivity and Case Align",
    labels={"smoothing_radius": "Smoothing radius", "mean_value": "Mean score", "series": "Metric"},
)
fig.update_layout(hovermode="x unified")
fig.show()

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
scores_path = OUTPUT_DIR / "imagenet_resnet50_smoothing_radius_scores_cosine_k25.csv"
summary_path = OUTPUT_DIR / "imagenet_resnet50_smoothing_radius_summary_cosine_k25.csv"
plot_long_path = OUTPUT_DIR / "imagenet_resnet50_smoothing_radius_plot_long_cosine_k25.csv"
config_path = OUTPUT_DIR / "imagenet_resnet50_smoothing_radius_config_cosine_k25.json"

scores_df.to_csv(scores_path, index=False)
summary_df.to_csv(summary_path, index=False)
plot_df.to_csv(plot_long_path, index=False)
with config_path.open("w", encoding="utf-8") as f:
    json.dump({
        "dataset": "imagenet_imagefolder_val",
        "imagenet_val_dir": str(IMAGENET_VAL_DIR),
        "model": "resnet50_pretrained",
        "methods": METHODS,
        "k": int(K),
        "problem_metric": PROBLEM_METRIC,
        "solution_metric": SOLUTION_METRIC,
        "smoothing_radii": [float(x) for x in SMOOTHING_RADII],
        "perturb_radius": float(PERTURB_RADIUS),
        "n_perturb_samples": int(N_PERTURB_SAMPLES),
        "retrieval_batch_size": int(RETRIEVAL_BATCH_SIZE),
        "max_retrieval_samples": int(MAX_RETRIEVAL_SAMPLES),
        "max_query_samples": int(MAX_QUERY_SAMPLES),
        "n_query_samples": int(len(query_images)),
        "retrieval_pool_size": int(len(retrieval_images)),
        "seed": int(SEED),
    }, f, indent=2)
print(scores_path)
print(summary_path)
print(plot_long_path)
print(config_path)